# 01 — Data Preparation

Steps covered (per project spec):
- **Step 2:** Load text CSV, drop empty rows, clean text
- **Step 6:** Verify image folder structure and class balance for the CNN

Inputs:
- `data/text/Combined Data.csv`
- `data/img/{train,test}/<emotion>/*.jpg`

In [ ]:
import pandas as pd
from pathlib import Path

from notebook_setup import add_project_to_path
from config import IMG_TEST, IMG_TRAIN, NORMAL_LABEL, TEXT_CSV
from text_clean import prepare_dataframe

ROOT = add_project_to_path()
print('Project root:', ROOT)
print('Text CSV   :', TEXT_CSV)
print('Train imgs :', IMG_TRAIN)
print('Test  imgs :', IMG_TEST)

## Text dataset

In [ ]:
df_raw = pd.read_csv(TEXT_CSV)
print('Raw shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_clean = prepare_dataframe(df_raw, 'statement')
df_clean['binary_label'] = (df_clean['status'] != NORMAL_LABEL).astype(int)
print('Clean shape:', df_clean.shape)
print('\nLabel distribution (binary):')
print(df_clean['binary_label'].value_counts().rename({0: 'Positive (Normal)', 1: 'Negative (Depression)'}))
print('\nOriginal status counts:')
print(df_clean['status'].value_counts())
df_clean.head()

In [ ]:
# Save cleaned dataframe for downstream notebooks
CLEAN_CSV = ROOT / 'data' / 'text' / 'cleaned.parquet'
df_clean[['clean_text', 'status', 'binary_label']].to_parquet(CLEAN_CSV, index=False)
print('Saved ->', CLEAN_CSV)

## Image dataset (FER-style 48×48)

In [ ]:
def folder_counts(root: Path) -> pd.DataFrame:
    rows = []
    for sub in sorted(p for p in root.iterdir() if p.is_dir()):
        n = sum(1 for _ in sub.iterdir())
        rows.append({'emotion': sub.name, 'count': n})
    return pd.DataFrame(rows)

train_counts = folder_counts(IMG_TRAIN)
test_counts = folder_counts(IMG_TEST)
print('Train:')
print(train_counts.to_string(index=False))
print('\nTest:')
print(test_counts.to_string(index=False))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(train_counts), figsize=(2 * len(train_counts), 2.2))
for ax, emo in zip(axes, train_counts['emotion']):
    sample = next((IMG_TRAIN / emo).iterdir())
    ax.imshow(Image.open(sample), cmap='gray')
    ax.set_title(emo, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

✅ Data ready. Continue to `02_train_nlp.ipynb`.